# Neural Network Month · Project 3
## Notebook 2 of 3 — The Graph Neural Network

In Project 1 every neuron saw the same inputs. In Project 2 filters slid across a grid of pixels. A **graph** has neither a fixed size nor a grid: each neuron has a different number of partners.

A **graph convolutional network (GCN)** solves this with one idea, repeated:

> **Each neuron updates itself by averaging the signals of its neighbours (and itself), then passing the result through a small neural network.**

Stack two layers and each neuron has absorbed information from its partners' partners. Everything else, training by backpropagation and a softmax at the end, is what you already know from earlier in the month.

**This notebook:** load the graph, build the GCN, train it on one fold, and draw how it works. Two figures. It runs on a normal CPU in about a minute.

## 1 · Setup, data, code

In [ ]:
import os, sys, glob, zipfile, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Patch
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

# One palette for the series: steel blue and lime green lead, like a lit-up brain scan.
P = {
    "steel":  "#4682B4",  # steel blue
    "lime":   "#8CC63F",  # lime green
    "teal":   "#2FB5A6",
    "violet": "#7A6BC4",
    "berry":  "#C2417E",
    "ink":    "#1E2B3A",  # text
    "paper":  "#F2F6F9",  # background
    "grid":   "#D3DCE4",
    "stone":  "#8896A5",  # neutral
    "rust":   "#B5452D",  # warnings (proxy graph)
}
# The smell pathway, nose to brain to body: lime -> teal -> steel -> violet -> berry
CLASSES = ["ALRN", "ALLN", "ALPN", "TOON", "DN"]
CLASS_NAMES = {"ALRN": "Receptor", "ALLN": "Local", "ALPN": "Projection", "TOON": "Third-Order", "DN": "Descending"}
CLASS_COLORS = {"ALRN": P["lime"], "ALLN": P["teal"], "ALPN": P["steel"], "TOON": P["violet"], "DN": P["berry"]}
MODEL_COLORS = {"Linear Regression": P["stone"], "Random Forest": P["steel"], "GCN": P["lime"]}
SEQ_CMAP = LinearSegmentedColormap.from_list("seq", [P["paper"], P["steel"], P["ink"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

def proxy_note(fig, y=-0.03):
    """Stamp figures that depend on a stand-in graph, so nobody mistakes them for real wiring."""
    if globals().get("EDGE_SOURCE", "") == "proxy":
        fig.text(0.5, y, "PROXY GRAPH: neighbours chosen by feature similarity, not real synapses",
                 ha="center", color=P["rust"], fontsize=10, fontweight="bold")

In [ ]:
# Where is the data? Download archive.zip from the Kaggle page and upload it to /content
# (the Colab file panel), or point ZIP_PATH at it. The cell also tries a Kaggle download.
ZIP_PATH = "/content/archive.zip"
DATA_DIR = "fly_data"

def locate_root(base):
    for p in glob.glob(f"{base}/**/S1_hemibrain_neuron_layers.csv", recursive=True):
        return os.path.dirname(p)
    return None

root = locate_root(DATA_DIR) if os.path.isdir(DATA_DIR) else None
if root is None and os.path.exists(ZIP_PATH):
    zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
    root = locate_root(DATA_DIR)
if root is None:
    try:
        import kagglehub
        root = locate_root(kagglehub.dataset_download("thedevastator/hemibrain-neuronal-connectome"))
    except Exception as e:
        print("Kaggle download did not work:", e)
if root is None:
    raise FileNotFoundError("Could not find the hemibrain CSVs. Upload the Kaggle zip and set ZIP_PATH.")
print("Data folder:", root)

In [ ]:
# ---- The only places we touch the data ----
N_FOLDS, SEED = 5, 42
FEATURES = ["pre", "post", "downstream", "voxels", "soma"]   # measured properties, no label information

def build_nodes(root):
    files = {"ALRN": "S3_hemibrain_ALRN_meta.csv", "ALLN": "S4_hemibrain_ALLN_meta.csv",
             "ALPN": "S5_hemibrain_ALPN_meta.csv", "TOON": "S6_hemibrain_TOON_meta.csv",
             "DN": "S7_hemibrain_DN_meta.csv"}
    cols = ["bodyid", "status"] + FEATURES
    nodes = pd.concat([pd.read_csv(os.path.join(root, f), usecols=cols).assign(cls=c) for c, f in files.items()],
                      ignore_index=True)
    # A neuron listed in two class tables has no single ground-truth label, so it is set aside.
    ambiguous = nodes.bodyid[nodes.bodyid.duplicated(keep=False)].unique()
    nodes = nodes[~nodes.bodyid.isin(ambiguous)].reset_index(drop=True)
    nodes["soma"] = nodes["soma"].astype(int)
    nodes["label"] = nodes.cls.map(CLASSES.index)
    # The authors' layer score is only used to DRAW the pathway, never as a feature or a target.
    layers = pd.read_csv(os.path.join(root, "S1_hemibrain_neuron_layers.csv")).rename(columns={"bodyId": "bodyid"})
    nodes = nodes.merge(layers[["bodyid", "layer_mean"]], on="bodyid", how="left")
    nodes["fold"] = -1
    for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(nodes, nodes.label)):
        nodes.loc[te, "fold"] = k
    return nodes, len(ambiguous)

In [ ]:
# ---- Where the wiring comes from ----
# Main route: the neuPrint server at Janelia (needs a free account token).
# Backup route: a clearly labelled PROXY graph built from the node table alone.
MIN_SYNAPSES      = 3                    # ignore connections weaker than this (a common guard against false synapse calls)
NEUPRINT_SERVER   = "neuprint.janelia.org"
NEUPRINT_DATASETS = ["hemibrain:v1.2.1", "hemibrain:v1.2", "hemibrain:v1.1"]   # the one that matches our IDs best is used
EDGES_CSV         = None                 # optional: path to your own edge file with columns bodyId_pre, bodyId_post, weight
USE_PROXY_IF_NEEDED = True

def get_token():
    """Looks for NEUPRINT_TOKEN in environment variables or Colab Secrets; in Colab, offers a hidden prompt."""
    tok = os.environ.get("NEUPRINT_TOKEN")
    if tok: return tok
    try:
        from google.colab import userdata
        tok = userdata.get("NEUPRINT_TOKEN")
        if tok: return tok
    except Exception:
        pass
    if "google.colab" in sys.modules:
        import getpass
        return getpass.getpass("neuPrint token (press Enter to skip and use the proxy graph): ").strip() or None
    return None

def fetch_edges_neuprint(ids, token):
    try:
        import neuprint
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q neuprint-python")
        import neuprint
    from neuprint import Client, fetch_adjacencies
    ids = [int(i) for i in ids]
    sample = [int(i) for i in np.random.default_rng(0).choice(ids, size=min(1000, len(ids)), replace=False)]
    best = None
    for ds in NEUPRINT_DATASETS:                      # the supplement and the server can differ by a data version
        try:
            client = Client(NEUPRINT_SERVER, dataset=ds, token=token)
            found = int(client.fetch_custom(f"MATCH (n:Neuron) WHERE n.bodyId IN {sample} RETURN count(n) AS n")["n"].iloc[0])
        except Exception as e:
            print(f"  {ds}: could not be queried ({str(e)[:90]})"); continue
        print(f"  {ds}: {found} of {len(sample)} sampled neuron IDs found")
        if best is None or found > best[1]: best = (ds, found, client)
    if best is None: raise RuntimeError("no neuPrint dataset could be queried")
    ds, found, client = best
    print(f"Using {ds}. Fetching connections among {len(ids)} neurons (this can take a few minutes)...")
    _, conn = fetch_adjacencies(sources=ids, targets=ids, min_total_weight=MIN_SYNAPSES,
                                omit_rois=True, weight_props=["weight"], client=client)
    return conn[["bodyId_pre", "bodyId_post", "weight"]], f"neuPrint {ds}"

def proxy_edges(nodes, k=10):
    """Backup graph: link each neuron to its k most similar neurons by measured features. NOT synapses."""
    from sklearn.neighbors import NearestNeighbors
    X = np.log1p(nodes[["pre", "post", "downstream", "voxels"]].astype(float))
    X["soma"] = nodes["soma"].astype(float)
    X = ((X - X.mean()) / X.std()).values
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(X)
    _, idx = nn.kneighbors(X)
    ids = nodes.bodyid.values
    rows = [(ids[i], ids[j], 1.0) for i in range(len(ids)) for j in idx[i, 1:]]
    return pd.DataFrame(rows, columns=["bodyId_pre", "bodyId_post", "weight"])

def get_edges(nodes):
    edges, source = None, None
    if EDGES_CSV:
        edges, source = pd.read_csv(EDGES_CSV), f"file {EDGES_CSV}"
    else:
        token = get_token()
        if token:
            try:
                edges, source = fetch_edges_neuprint(nodes.bodyid.values, token)
            except Exception as e:
                print("neuPrint route failed:", repr(e)[:300])
        else:
            print("No neuPrint token found.")
    if edges is None:
        if not USE_PROXY_IF_NEEDED: raise RuntimeError("No edges available.")
        print("\n*** Falling back to the PROXY graph: neighbours are chosen by feature similarity, NOT real synapses. ***\n")
        return proxy_edges(nodes), "proxy"
    # Necessary tidying: keep our neurons only, drop self-connections, merge repeated pairs, apply the weight floor
    keep = set(nodes.bodyid)
    edges = edges[edges.bodyId_pre.isin(keep) & edges.bodyId_post.isin(keep) & (edges.bodyId_pre != edges.bodyId_post)]
    edges = edges.groupby(["bodyId_pre", "bodyId_post"], as_index=False)["weight"].sum()
    return edges[edges.weight >= MIN_SYNAPSES].reset_index(drop=True), source

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import scipy.sparse as sp
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

def make_features(nodes):
    """Log-scale the heavy-tailed counts, then standardise. Uses no labels."""
    X = np.log1p(nodes[["pre", "post", "downstream", "voxels"]].astype(float).clip(lower=0))
    X["soma"] = nodes["soma"].astype(float)
    return ((X - X.mean()) / X.std()).values.astype(np.float32)

def build_adjacency(edges, id_to_idx, n):
    """Normalised adjacency A_hat. Averaging over neighbours = multiplying by this matrix."""
    src = edges.bodyId_pre.map(id_to_idx).values
    dst = edges.bodyId_post.map(id_to_idx).values
    w = np.log1p(edges.weight.values)                       # tame the heavy-tailed synapse counts
    A = sp.coo_matrix((w, (src, dst)), shape=(n, n)).tocsr()
    A = A + A.T                                             # synapses are directed; the GCN listens both ways
    A = A + sp.eye(n)                                       # every neuron also keeps its own signal
    dinv = sp.diags(np.asarray(A.sum(1)).ravel() ** -0.5)
    A_hat = (dinv @ A @ dinv).tocoo()
    idx = torch.tensor(np.vstack([A_hat.row, A_hat.col]), dtype=torch.long)
    return torch.sparse_coo_tensor(idx, torch.tensor(A_hat.data, dtype=torch.float32), (n, n)).coalesce()

class GCN(nn.Module):
    """Two graph-convolution layers: each neuron averages its neighbours' signals, then a small network transforms them."""
    def __init__(self, n_in, n_hidden, n_out, dropout=0.5):
        super().__init__()
        self.lin1, self.lin2, self.drop = nn.Linear(n_in, n_hidden), nn.Linear(n_hidden, n_out), nn.Dropout(dropout)
    def hidden(self, X, A):                                 # layer 1: gather direct partners
        return F.relu(torch.sparse.mm(A, self.lin1(self.drop(X))))
    def forward(self, X, A):                                # layer 2: gather partners of partners
        return torch.sparse.mm(A, self.lin2(self.drop(self.hidden(X, A))))

def train_gcn(X, A, y, tr_idx, va_idx, n_classes, epochs=200, lr=0.02, wd=5e-4, hidden=32, seed=0):
    torch.manual_seed(seed)
    model = GCN(X.shape[1], hidden, n_classes)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    counts = torch.bincount(y[tr_idx], minlength=n_classes).float()
    cw = (len(tr_idx) / (n_classes * counts.clamp(min=1)))      # rarer classes count more
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for _ in range(epochs):
        model.train()
        loss = F.cross_entropy(model(X, A)[tr_idx], y[tr_idx], weight=cw)
        opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            out = model(X, A)
            hist["train_loss"].append(F.cross_entropy(out[tr_idx], y[tr_idx], weight=cw).item())
            hist["val_loss"].append(F.cross_entropy(out[va_idx], y[va_idx], weight=cw).item())
            hist["train_acc"].append((out[tr_idx].argmax(1) == y[tr_idx]).float().mean().item())
            hist["val_acc"].append((out[va_idx].argmax(1) == y[va_idx]).float().mean().item())
    return model, hist

def score_all(y_true, probs, n_classes):
    """Five scores per model. Precision, recall and F1 are macro-averaged (every class counts equally).
    R-squared compares the model's class probabilities with the true one-hot answer."""
    pred = probs.argmax(1)
    Y = np.eye(n_classes)[y_true]
    r2 = 1 - ((Y - probs) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum()
    return {"Accuracy": accuracy_score(y_true, pred),
            "F1": f1_score(y_true, pred, average="macro", zero_division=0),
            "Precision": precision_score(y_true, pred, average="macro", zero_division=0),
            "Recall": recall_score(y_true, pred, average="macro", zero_division=0),
            "R²": r2}

In [ ]:
if os.path.exists("fly_nodes.csv") and os.path.exists("fly_edges.csv"):      # saved by Notebook 1
    nodes, edges = pd.read_csv("fly_nodes.csv"), pd.read_csv("fly_edges.csv")
    EDGE_SOURCE = open("fly_edge_source.txt").read().strip()
else:                                                                         # otherwise rebuild the same objects
    nodes, _ = build_nodes(root)
    edges, EDGE_SOURCE = get_edges(nodes)
print(f"{len(nodes)} neurons | {len(edges)} connections | edge source: {EDGE_SOURCE}")

## 2 · From wiring to a matrix

Averaging over neighbours can be written as one matrix multiplication. We build a matrix `A_hat` where row *i* holds neuron *i*'s neighbours, with three small choices:

- **Direction ignored for averaging.** Synapses are directed, but a neuron can learn from partners on both sides, so we make the graph symmetric.
- **Self-loops added.** Each neuron keeps its own signal along with its neighbours'.
- **Normalised by degree.** So a neuron with 500 partners does not drown out one with 5.

Synapse counts are log-scaled so a few very strong connections do not dominate.

In [ ]:
X_np = make_features(nodes)
y_np = nodes.label.values
id_to_idx = {b: i for i, b in enumerate(nodes.bodyid)}
A = build_adjacency(edges, id_to_idx, len(nodes))
X, y = torch.tensor(X_np), torch.tensor(y_np, dtype=torch.long)
n_classes = len(CLASSES)
print(f"Feature matrix: {tuple(X.shape)} | adjacency: {tuple(A.shape)} with {A._nnz():,} non-zero entries")

## 3 · The model

Read `GCN` top to bottom. `hidden` is layer 1: multiply by `A` (gather partners), then a learned transformation and a ReLU. `forward` repeats the gather and transforms down to five scores, one per class. It has only a few hundred parameters, far fewer than the CNN, because the graph does much of the work.

In [ ]:
print("Trainable parameters:", sum(p.numel() for p in GCN(X.shape[1], 32, n_classes).parameters()))

## 4 · Train on one fold

We hold out fold 1 of our stratified 5-fold split, train on the other four, and track loss and accuracy on the held-out neurons.

One honest detail: this is **transductive** learning. The whole graph and all neurons' *measurements* are visible while training, but only the training neurons' *labels* are used. That is the standard setup for node classification.

Class weights make errors on the rare classes cost more, so the model cannot win by ignoring them.

In [ ]:
tr_idx = torch.tensor(np.where(nodes.fold.values != 0)[0]); va_idx = torch.tensor(np.where(nodes.fold.values == 0)[0])
t0 = time.time()
model, hist = train_gcn(X, A, y, tr_idx, va_idx, n_classes)
print(f"Trained in {time.time()-t0:.0f}s | held-out accuracy {hist['val_acc'][-1]:.3f} | edge source: {EDGE_SOURCE}")

## 5 · Figure 5 — how the GCN works

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.4)); ax.axis("off"); ax.grid(False)
ax.set_xlim(0, 13); ax.set_ylim(0, 6.4)

def box(x, y, w, h, color, title, sub, text_color="white"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04,rounding_size=0.18", fc=color, ec=P["ink"], lw=1.6))
    ax.text(x + w / 2, y + h * 0.63, title, ha="center", va="center", fontsize=13, fontweight="bold", color=text_color)
    ax.text(x + w / 2, y + h * 0.28, sub, ha="center", va="center", fontsize=9.5, color=text_color)

def arrow(x0, y0, x1, y1, color=P["ink"], lw=2.0):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", color=color, lw=lw))

# top: the pipeline
box(0.3, 4.3, 2.7, 1.5, P["lime"], "5 features", "pre, post, downstream,\nsize, soma", text_color=P["ink"])
box(3.7, 4.3, 2.7, 1.5, P["steel"], "GCN layer 1", "average partners\n32 hidden units")
box(7.0, 4.3, 2.7, 1.5, P["steel"], "GCN layer 2", "average again\n5 class scores")
box(10.3, 4.3, 2.5, 1.5, P["violet"], "Class", "softmax: one probability\nper class")
for x0, x1 in [(3.0, 3.7), (6.4, 7.0), (9.7, 10.3)]:
    arrow(x0, 5.05, x1, 5.05)

def star(cx, cy, centre_color, nbr_colors, label):
    ang = np.linspace(0, 2 * np.pi, len(nbr_colors), endpoint=False) + 0.5
    for a, c in zip(ang, nbr_colors):
        nx_, ny_ = cx + 1.05 * np.cos(a), cy + 0.95 * np.sin(a)
        arrow(nx_ - 0.18 * np.cos(a), ny_ - 0.18 * np.sin(a), cx + 0.3 * np.cos(a), cy + 0.3 * np.sin(a), color=P["stone"], lw=1.6)
        ax.scatter(nx_, ny_, s=330, color=c, edgecolor=P["ink"], linewidth=1.3, zorder=3)
    ax.scatter(cx, cy, s=620, color=centre_color, edgecolor=P["ink"], linewidth=1.6, zorder=4)
    ax.text(cx, cy - 1.55, label, ha="center", va="center", fontsize=10.5)

cc = CLASS_COLORS
star(5.05, 2.0, cc["ALPN"], [cc["ALRN"], cc["ALLN"], cc["TOON"], cc["ALRN"], cc["ALPN"]], "1 hop: direct partners")
star(8.35, 2.0, cc["ALPN"], [cc["TOON"], cc["TOON"], cc["ALPN"], cc["DN"]], "2 hops: partners of partners")
# bottom left: the wiring as input
pts = np.array([[1.0, 2.5], [2.2, 1.4], [0.8, 1.1], [2.0, 2.7], [1.5, 1.9]])
for a, b in [(0, 4), (4, 1), (2, 4), (3, 4), (0, 3), (2, 1)]:
    ax.plot(*zip(pts[a], pts[b]), color=P["stone"], lw=1.6, zorder=1)
for p, c in zip(pts, [cc["ALRN"], cc["TOON"], cc["ALLN"], cc["DN"], cc["ALPN"]]):
    ax.scatter(*p, s=330, color=c, edgecolor=P["ink"], linewidth=1.3, zorder=3)
ax.text(1.55, 0.45, "the wiring diagram", ha="center", fontsize=10.5)
# bottom right: output classes
for i, c in enumerate(CLASSES):
    ax.scatter(10.55, 3.1 - 0.62 * i, s=230, color=CLASS_COLORS[c], edgecolor=P["ink"], linewidth=1.2)
    ax.text(10.85, 3.1 - 0.62 * i, CLASS_NAMES[c], va="center", fontsize=10.5)
arrow(1.55, 3.0, 1.55, 4.2, color=P["stone"]); arrow(5.05, 3.45, 5.05, 4.2, color=P["stone"]); arrow(8.35, 3.45, 8.35, 4.2, color=P["stone"])
ax.set_title("How Our GCN Works", fontsize=16, pad=6)
fig.tight_layout(); proxy_note(fig, y=0.0); save_fig(fig, "05_how_our_gcn_works"); plt.show()

Read it left to right. The features of each neuron go in, layer 1 mixes in its direct partners, layer 2 mixes in their partners, and a final step turns scores into one probability per class. The small star diagrams show the **gather** step: the centre neuron's new value is a weighted average of everything pointing at it.

## 6 · Figure 6 — learning curves

In [ ]:
ep = np.arange(1, len(hist["train_loss"]) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, key, yl in [(a1, "loss", "Weighted cross-entropy"), (a2, "acc", "Accuracy")]:
    ax.plot(ep, hist[f"train_{key}"], color=P["lime"], linewidth=2.8, label="Training neurons")
    ax.plot(ep, hist[f"val_{key}"], color=P["steel"], linewidth=2.8, label="Held-out neurons")
    ax.set_xlabel("Epoch"); ax.set_ylabel(yl); ax.legend(frameon=False)
a1.set_title("Loss"); a2.set_title("Accuracy")
fig.suptitle("Training and Validation Curves", y=1.02)
fig.tight_layout(); proxy_note(fig, y=-0.04); save_fig(fig, "06_training_curves"); plt.show()

**How to read it.** Both lines should improve together. A widening gap, with training better and held-out worse, would mean the network is memorising individual neurons instead of learning the pattern. Accuracy here is the plain overall score, so it will look flattering on this imbalanced data; Notebook 3 uses fairer measures.

**Next:** Notebook 3 trains five GCNs plus two classic baselines, and compares them.

In [ ]:
# Optional: download this notebook's figures
try:
    import shutil
    from google.colab import files
    shutil.make_archive("nb2_figures", "zip", "figures")
    files.download("nb2_figures.zip")
except ImportError:
    print("Figures are saved in the 'figures' folder.")